## I. Setup Software and Some Libraries

We first need to set the working environment and the path to the dataset.

In [81]:
import sys

SOFTWARE_DIR = '/app/data/spine/' # Change this path to your software install
DATA_DIR = '/app/data/' # Change this path if you are not on SDF (see main README)

# Set software directory
sys.path.append(SOFTWARE_DIR)

import numpy as np
import math
import pandas as pd
from collections import OrderedDict

from scipy.spatial.distance import cdist

Now pass the analysis configuration.

In [85]:
import yaml
from spine.driver import Driver

# file_name = 'packet-0050017-2024_07_09_00_04_33_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_14_34_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
file_name = 'packet-0050017-2024_07_09_00_24_35_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_34_36_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_44_37_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_54_38_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_04_39_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_14_40_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_24_41_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_34_42_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_44_44_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_54_45_CDT.LARCV_spine.h5' # latest spine reprocessing (v6 - with steven's filter)


# file_name = 'packet-0050017-2024_07_09_00_04_33_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_14_34_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_24_35_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_34_36_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_44_37_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_00_54_38_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_04_39_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_14_40_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_24_41_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_34_42_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_44_44_CDT.SPINE.h5' # (v4 - no steven's filter)
# file_name = 'packet-0050017-2024_07_09_01_54_45_CDT.SPINE.h5' # (v4 - no steven's filter) 


DATA_PATH = DATA_DIR + file_name

cfg = '''
base:
  iterations: 10
io:
  reader:
    name: hdf5 # Type of input reader
    file_keys: DATA_PATH # Path to the data file
    create_run_map: true # Creates a map between run/event pairs and entry
    skip_unknown_attrs: true # Allows to load files made with an older SPINE
build:
  mode: reco # Build only reconstructed objects (no truth information)
  fragments: false # Do not build fragment objects (not stored by default)
  particles: true # Build `RecoParticle`/`TruthParticle` objects
  interactions: true # Build `RecoInteraction`/`TruthInteraction` objects

'''.replace('DATA_PATH', file_name)

cfg = yaml.safe_load(cfg)
driver = Driver(cfg)



 ██████████   ██████████    ███   ███       ██   ███████████
███        █  ██       ███   █    █████     ██   ██         
  ████████    ██       ███  ███   ██  ████  ██   ██████████ 
█        ███  ██████████     █    ██     █████   ██         
 ██████████   ██            ███   ██       ███   ███████████

Release version: 0.2.2

$CUDA_VISIBLE_DEVICES=

Configuration processed at: Linux d50fe8aa05a6 5.15.167.4-microsoft-standard-WSL2 #1 SMP Tue Nov 5 00:21:55 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux

base: {iterations: 10, seed: 1739218728}
io:
  reader: {name: hdf5, file_keys: packet-0050017-2024_07_09_00_24_35_CDT.LARCV_spine.h5,
    create_run_map: true, skip_unknown_attrs: true}
build: {mode: reco, fragments: false, particles: true, interactions: true}

Will load 1 file(s):
  - packet-0050017-2024_07_09_00_24_35_CDT.LARCV_spine.h5

Total number of entries in the file(s): 456

Total number of entries selected: 456



We are doing following here:
- Map ND-LAr events to their corresponding `entry` numbers in SPINE files using the `run_info` field.
- Initialize the driver with this configuration.
- Retrieves and analyzes the `reco_particles` and `reco_interactions` data structures. It provides a count of each structure and collects the unique **Reco Interaction IDs** from the `reco_particles` dataset.

In [87]:
import h5py
from collections import defaultdict

# Target Event (Update this as needed)
target_event = 378  # Example: Target Event Number

# Define the mapping of PID to particle names
pid_to_name = {
    0: "Photon",
    1: "Electron",
    2: "Muon",
    3: "Pion",
    4: "Proton",
    5: "Kaon"
}

# Locate the entry corresponding to the target event
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    target_entry = None

    for idx, (run, subrun, event) in enumerate(run_info):
        if event == target_event:
            target_entry = idx
            print(f"Found target event {target_event} at entry {idx}")
            break
    else:
        print(f"Event {target_event} not found in run_info.")
        target_entry = None

# Process the entry if found
if target_entry is not None:
    data = driver.process(entry=target_entry)
    reco_particles = data['reco_particles']
    reco_interactions = data['reco_interactions']

    print(f"Total Reconstructed Particles for Event {target_event}: {len(reco_particles)}")
    print(f"Total Reconstructed Interactions for Event {target_event}: {len(reco_interactions)}\n")

    # Group particles by their interaction IDs
    all_reco_interactions = {}
    particle_summary = defaultdict(lambda: {"Primary": 0, "Non-Primary": 0})

    for interaction in reco_interactions:
        interaction_id = interaction.id
        interaction_particles = [p for p in reco_particles if p.interaction_id == interaction_id]
        all_reco_interactions[interaction_id] = interaction_particles

        for particle in interaction_particles:
            pid = particle.pid
            if particle.is_primary:
                particle_summary[pid]["Primary"] += 1
            else:
                particle_summary[pid]["Non-Primary"] += 1

        print(f"Interaction ID {interaction_id}: {len(interaction_particles)} particles")

    # Print particle summary across all interactions
    print("\nSummary of particles across all interactions:")
    total_showers = {"Primary": 0, "Non-Primary": 0}
    total_tracks = {"Primary": 0, "Non-Primary": 0}

    for pid, counts in sorted(particle_summary.items()):
        primary_count = counts["Primary"]
        non_primary_count = counts["Non-Primary"]
        particle_name = pid_to_name.get(pid, "Unknown")
        print(f"{particle_name}: {primary_count + non_primary_count} (Primary: {primary_count}; Non-Primary: {non_primary_count})")

        # Categorize as shower or track
        if pid in {0, 1}:  # Photon or Electron
            total_showers["Primary"] += primary_count
            total_showers["Non-Primary"] += non_primary_count
        elif pid in {2, 3, 4, 5}:  # Muon, Pion, Proton, Kaon
            total_tracks["Primary"] += primary_count
            total_tracks["Non-Primary"] += non_primary_count

    # Print shower and track summary
    total_showers_count = total_showers["Primary"] + total_showers["Non-Primary"]
    total_tracks_count = total_tracks["Primary"] + total_tracks["Non-Primary"]

    print("\nShower Summary:")
    print(f"Total Showers: {total_showers_count} (Primary: {total_showers['Primary']}; Non-Primary: {total_showers['Non-Primary']})")

    print("\nTrack Summary:")
    print(f"Total Tracks: {total_tracks_count} (Primary: {total_tracks['Primary']}; Non-Primary: {total_tracks['Non-Primary']})")


Found target event 378 at entry 378
Total Reconstructed Particles for Event 378: 17
Total Reconstructed Interactions for Event 378: 2

Interaction ID 0: 14 particles
Interaction ID 1: 3 particles

Summary of particles across all interactions:
Photon: 3 (Primary: 2; Non-Primary: 1)
Electron: 6 (Primary: 3; Non-Primary: 3)
Muon: 3 (Primary: 3; Non-Primary: 0)
Pion: 1 (Primary: 1; Non-Primary: 0)
Proton: 4 (Primary: 2; Non-Primary: 2)

Shower Summary:
Total Showers: 9 (Primary: 5; Non-Primary: 4)

Track Summary:
Total Tracks: 8 (Primary: 6; Non-Primary: 2)


In [88]:
from spine.vis import Drawer
import plotly.io as pio
import os

# Ensure `reco_particles` and `reco_interactions` are available
if not reco_particles or not reco_interactions:
    raise ValueError("No reconstructed particles or interactions found. Please run Cell 1 first.")

# Extract the specific part ("00_04_33") from the filename
file_segment = '_'.join(os.path.basename(DATA_PATH).split('_')[3:6])

# Create the images directory if it doesn't exist
output_dir = "images"
os.makedirs(output_dir, exist_ok=True)

# Initialize the drawer with all reconstructed particles
drawer = Drawer(data, draw_mode='reco', detector='2x2')

# Draw a plot of all particle instances across all interactions
print('\n>>> All Reconstructed Interactions <<<')
fig_particles = drawer.get(
    'particles',
    attr='pid',
    draw_end_points=True,
    draw_vertices=True,
    split_traces=True  # Ensure each particle gets a separate legend entry
)

# Save the combined particle plot as HTML and PNG in the images folder
html_filename = os.path.join(output_dir, f"sandbox_{file_segment}_entry_{target_entry}_event_{target_event}_SPINE.html")
png_filename = os.path.join(output_dir, f"sandbox_{file_segment}_entry_{target_entry}_event_{target_event}_SPINE.png")

# Save as HTML
pio.write_html(fig_particles, file=html_filename)
print(f"Combined Particle Plot saved as HTML: {html_filename}")

# Save as PNG
pio.write_image(fig_particles, file=png_filename, format='png', width=1000, height=800)
print(f"Combined Particle Plot saved as PNG: {png_filename}")

# Display the visualization
fig_particles.show()



>>> All Reconstructed Interactions <<<
Combined Particle Plot saved as HTML: images/sandbox_00_24_35_entry_378_event_378_SPINE.html
Combined Particle Plot saved as PNG: images/sandbox_00_24_35_entry_378_event_378_SPINE.png
